Gate 1 → Bronze

Mỗi lô: qua cổng → đã có trong sổ thì bỏ qua → đọc CSV **mọi cột string** → gắn cột kỹ thuật →
**ghi Bronze (append)** → **rồi mới** ghi sổ `loaded_batches.csv`.
Bronze ghi vào `_sandbox/bronze`, sổ ở `_sandbox/control`

In [1]:
import csv, json, logging, shutil, sys
from pathlib import Path

import pandas as pd

# Thư mục gốc dự án = nơi có src/config.py (kernel có thể mở với cwd bất kỳ)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
for p in (ROOT, ROOT / "tests"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import bronze, config, validate

logging.basicConfig(level=logging.INFO, format="%(levelname)-5s %(name)-7s %(message)s",
                    stream=sys.stdout, force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)
pd.set_option("display.max_columns", 40)

SANDBOX = ROOT / "notebook" / "_sandbox"             # mọi thứ notebook ghi ra nằm ở đây
SB_BRONZE = SANDBOX / "bronze" / "business_valuation"
SB_CONTROL = SANDBOX / "control" / "loaded_batches.csv"
MIXED = SANDBOX / "mixed"                            # landing thử: 1 lô tốt + 1 lô hỏng
SRC = config.LANDING_DIR / "as_of_month=2026-06"     # lô dùng để soi, làm hỏng, gửi lại
RESEND = SANDBOX / "resend" / SRC.name               # lô gửi lại

EXPECTED = {"bronze": 52_791}

Hàm tiện ích dùng trong notebook này:

In [2]:
def fresh(*paths: Path) -> None:
    """Xoá các thư mục/file con của sandbox — notebook chỉ dọn phần của mình."""
    for p in paths:
        if SANDBOX not in p.parents:
            raise ValueError(f"{p} nằm ngoài sandbox — không xoá")
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists():
            p.unlink()
    SANDBOX.mkdir(parents=True, exist_ok=True)

fresh(SB_BRONZE.parent, SB_CONTROL.parent, MIXED, RESEND.parent)
fresh

<function __main__.fresh(*paths: pathlib.Path) -> None>

In [3]:
def show(df, n: int = 10) -> pd.DataFrame:
    """DataFrame Spark -> pandas để notebook hiển thị thành bảng."""
    return df.limit(n).toPandas()

def edit_manifest(batch_dir: Path, **changes) -> None:
    m = validate.load_manifest(batch_dir)
    m.update(changes)
    (batch_dir / "_manifest.json").write_text(json.dumps(m, ensure_ascii=False, indent=2),
                                              encoding="utf-8")

def data_path(batch_dir: Path) -> Path:
    return batch_dir / validate.load_manifest(batch_dir)["file_name"]

def tamper_one_cell(batch_dir: Path) -> None:
    """Sửa tay một ô sau khi bàn giao (số dòng, header không đổi) -> chỉ checksum lệch."""
    f = data_path(batch_dir)
    f.write_bytes(f.read_bytes().replace(b",VND,", ",VNĐ,".encode("utf-8"), 1))

In [4]:
from pyspark.sql import functions as F
from src.spark import get_spark

spark = get_spark("02_bronze")
print("Spark", spark.version)

Spark 3.5.9


## Đọc một lô từ gate1

In [5]:
manifest = validate.load_manifest(SRC)
data_file = SRC / manifest["file_name"]

# cache(): Spark không cho truy vấn CHỈ cột _corrupt_record trên CSV thô chưa cache
one = bronze.read_batch(spark, data_file, manifest).cache()
one.printSchema()
n_corrupt = one.filter(F.col(bronze.CORRUPT_COL).isNotNull()).count()
print(f"{one.count()} dòng · vỡ cột: {n_corrupt} · mọi cột string: "
      f"{all(t == 'string' for _, t in one.dtypes)}")
show(one, 5)

root
 |-- ma_nghiep_vu: string (nullable = true)
 |-- nhom_xe: string (nullable = true)
 |-- hang_xe: string (nullable = true)
 |-- ma_dong_xe: string (nullable = true)
 |-- ma_kieu_dang: string (nullable = true)
 |-- so_cho_ngoi: string (nullable = true)
 |-- nam_san_xuat: string (nullable = true)
 |-- gia_tri_dinh_gia: string (nullable = true)
 |-- ngay_duyet: string (nullable = true)
 |-- hieu_luc_tu: string (nullable = true)
 |-- hieu_luc_den: string (nullable = true)
 |-- ky_bao_cao: string (nullable = true)
 |-- ma_dinh_gia: string (nullable = true)
 |-- don_vi_tinh: string (nullable = true)
 |-- loai_dinh_gia: string (nullable = true)
 |-- as_of_month: string (nullable = true)
 |-- _corrupt_record: string (nullable = true)

3394 dòng · vỡ cột: 0 · mọi cột string: True


,ma_nghiep_vu,nhom_xe,hang_xe,ma_dong_xe,ma_kieu_dang,so_cho_ngoi,nam_san_xuat,gia_tri_dinh_gia,ngay_duyet,hieu_luc_tu,hieu_luc_den,ky_bao_cao,ma_dinh_gia,don_vi_tinh,loai_dinh_gia,as_of_month,_corrupt_record
0,070104,KKD1-1,None,HYU15014,None,5,2020,260000000,2026-06-19,2026-07-02,2027-07-02,2026-06,BV-0e7043455756e930,VND,used,2026-06,None
1,070104,KKD1-1,None,FOR03021,None,5,2019,0,2026-06-13,2026-03-12,2026-03-12,2026-06,BV-9099de8997350b21,VND,used,2026-06,None
2,070104,KKD1-2,None,HYU01035,None,7,2018,630000000,2026-06-01,2026-06-04,2027-06-04,2026-06,BV-125d50e3bf8bcaff,VND,used,2026-06,None
3,070104,KKD1-2,None,VIN00303,None,7,2021,700000000,2026-06-04,2026-06-05,2027-06-05,2026-06,BV-581c745a2452d5a7,VND,used,2026-06,None
4,070104,KKD1-1,None,HON03009,None,5,2022,410000000,2026-06-01,2026-06-01,2027-06-01,2026-06,BV-667ad93ca9dd6b2a,VND,used,2026-06,None


## Nạp data từ gate1 vào bronze

In [6]:
batches = validate.list_batches(config.LANDING_DIR)
res_bronze = bronze.run(spark, config.LANDING_DIR, SB_BRONZE, SB_CONTROL)
print({k: len(v) for k, v in res_bronze.items()})
pd.read_csv(SB_CONTROL)                              # sổ lô đã nạp

INFO  bronze  NẠP     as_of_month=2025-01    3119 dòng  (batch_id=bvv_backfill_202501_01)
INFO  bronze  NẠP     as_of_month=2025-02    1982 dòng  (batch_id=bvv_backfill_202502_01)
INFO  bronze  NẠP     as_of_month=2025-03    2650 dòng  (batch_id=bvv_backfill_202503_01)
INFO  bronze  NẠP     as_of_month=2025-04    2417 dòng  (batch_id=bvv_backfill_202504_01)
INFO  bronze  NẠP     as_of_month=2025-05    2422 dòng  (batch_id=bvv_backfill_202505_01)
INFO  bronze  NẠP     as_of_month=2025-06    2507 dòng  (batch_id=bvv_backfill_202506_01)
INFO  bronze  NẠP     as_of_month=2025-07    2775 dòng  (batch_id=bvv_backfill_202507_01)
INFO  bronze  NẠP     as_of_month=2025-08    2333 dòng  (batch_id=bvv_backfill_202508_01)
INFO  bronze  NẠP     as_of_month=2025-09    2279 dòng  (batch_id=bvv_backfill_202509_01)
INFO  bronze  NẠP     as_of_month=2025-10    3418 dòng  (batch_id=bvv_backfill_202510_01)
INFO  bronze  NẠP     as_of_month=2025-11    3113 dòng  (batch_id=bvv_backfill_202511_01)
INFO  bron

,batch_id,as_of_month,row_count,ingest_ts,source_file
0,bvv_backfill_202501_01,2025-01,3119,2026-09-29T17:04:42+07:00,data/landing/business_vehicle_valuation/as_of_...
1,bvv_backfill_202502_01,2025-02,1982,2026-09-29T17:04:43+07:00,data/landing/business_vehicle_valuation/as_of_...
2,bvv_backfill_202503_01,2025-03,2650,2026-09-29T17:04:44+07:00,data/landing/business_vehicle_valuation/as_of_...
3,bvv_backfill_202504_01,2025-04,2417,2026-09-29T17:04:44+07:00,data/landing/business_vehicle_valuation/as_of_...
4,bvv_backfill_202505_01,2025-05,2422,2026-09-29T17:04:45+07:00,data/landing/business_vehicle_valuation/as_of_...
5,bvv_backfill_202506_01,2025-06,2507,2026-09-29T17:04:45+07:00,data/landing/business_vehicle_valuation/as_of_...
6,bvv_backfill_202507_01,2025-07,2775,2026-09-29T17:04:45+07:00,data/landing/business_vehicle_valuation/as_of_...
7,bvv_backfill_202508_01,2025-08,2333,2026-09-29T17:04:46+07:00,data/landing/business_vehicle_valuation/as_of_...
8,bvv_backfill_202509_01,2025-09,2279,2026-09-29T17:04:46+07:00,data/landing/business_vehicle_valuation/as_of_...
9,bvv_backfill_202510_01,2025-10,3418,2026-09-29T17:04:46+07:00,data/landing/business_vehicle_valuation/as_of_...


## Kiểm tra Bronze khớp manifest từng lô

In [7]:
bz = spark.read.parquet(SB_BRONZE.as_posix())
per_batch = bz.groupBy("as_of_month", "_batch_id").count().orderBy("as_of_month").toPandas()
per_batch["manifest"] = [int(validate.load_manifest(config.LANDING_DIR / f"as_of_month={m}")["record_count"])
                         for m in per_batch["as_of_month"]]
per_batch["khớp"] = per_batch["count"] == per_batch["manifest"]

assert per_batch["khớp"].all(), "Số dòng Bronze lệch manifest"
print("Tổng Bronze:", per_batch["count"].sum(), "· tài liệu:", EXPECTED["bronze"])
print("Cột kỹ thuật:", [c for c in bz.columns if c.startswith("_")])
per_batch

Tổng Bronze: 52791 · tài liệu: 52791
Cột kỹ thuật: ['_corrupt_record', '_batch_id', '_source_file', '_ingest_ts']


,as_of_month,_batch_id,count,manifest,khớp
0,2025-01,bvv_backfill_202501_01,3119,3119,True
1,2025-02,bvv_backfill_202502_01,1982,1982,True
2,2025-03,bvv_backfill_202503_01,2650,2650,True
3,2025-04,bvv_backfill_202504_01,2417,2417,True
4,2025-05,bvv_backfill_202505_01,2422,2422,True
5,2025-06,bvv_backfill_202506_01,2507,2507,True
6,2025-07,bvv_backfill_202507_01,2775,2775,True
7,2025-08,bvv_backfill_202508_01,2333,2333,True
8,2025-09,bvv_backfill_202509_01,2279,2279,True
9,2025-10,bvv_backfill_202510_01,3418,3418,True


## Idempotent: 
Lô đã có trong sổ bị **bỏ qua**, Bronze giữ nguyên số dòng.

In [8]:
n_before = bz.count()
res_again = bronze.run(spark, config.LANDING_DIR, SB_BRONZE, SB_CONTROL)
n_after = spark.read.parquet(SB_BRONZE.as_posix()).count()
print({k: len(v) for k, v in res_again.items()}, f"· Bronze {n_before} → {n_after}")

assert res_again["loaded"] == [] and len(res_again["skipped"]) == len(batches)
assert n_before == n_after

INFO  bronze  Bronze: 0 lô nạp mới, bỏ qua 18 lô đã nạp, 0 bị từ chối, 0 lỗi
{'loaded': 0, 'skipped': 18, 'rejected': 0, 'failed': 0} · Bronze 52791 → 52791


## Một lô hỏng không chặn lô khác

Landing thử gồm lô `2026-05` tốt + bản `2026-06` bị sửa tay một ô.

In [9]:
shutil.copytree(config.LANDING_DIR / "as_of_month=2026-05", MIXED / "landing" / "as_of_month=2026-05")
bad = MIXED / "landing" / SRC.name
shutil.copytree(SRC, bad)
tamper_one_cell(bad)

res_mixed = bronze.run(spark, MIXED / "landing", MIXED / "bronze", MIXED / "loaded_batches.csv")
print(res_mixed)
assert len(res_mixed["loaded"]) == 1 and res_mixed["rejected"] == [SRC.name]

INFO  bronze  NẠP     as_of_month=2026-05    3048 dòng  (batch_id=bvv_backfill_202605_01)
ERROR bronze  REJECT  as_of_month=2026-06
ERROR bronze          CHECKSUM_MISMATCH      sha256 796d8c4bba51… ≠ manifest 0021a3651a17…
INFO  bronze  Bronze: 1 lô nạp mới, bỏ qua 0 lô đã nạp, 1 bị từ chối, 0 lỗi
{'loaded': ['bvv_backfill_202605_01'], 'skipped': [], 'rejected': ['as_of_month=2026-06'], 'failed': []}


## Gửi lại lô mới — Bronze append-only

- Gửi lại lô `2026-06` với `batch_id` mới (dữ liệu giữ nguyên). 
- Gọi thẳng `ingest_batch` để mô phỏng.
- Bronze giờ có **hai bản** tháng 6 — bản cũ không bị ghi đè, vẫn còn để truy vết. 

In [10]:
shutil.copytree(SRC, RESEND)
edit_manifest(RESEND, batch_id="bvv_202606_02")      # lô gửi lại = batch_id mới
assert validate.validate_batch(RESEND) == []

n_resend = bronze.ingest_batch(spark, RESEND, SB_BRONZE, SB_CONTROL)
bz = spark.read.parquet(SB_BRONZE.as_posix())
print(f"Nạp thêm {n_resend} dòng · Bronze giờ có {bz.count()} dòng")
show(bz.filter("as_of_month = '2026-06'").groupBy("_batch_id", "_ingest_ts").count())

Nạp thêm 3394 dòng · Bronze giờ có 56185 dòng


,_batch_id,_ingest_ts,count
0,bvv_202606_01,2026-09-29 17:04:48,3394
1,bvv_202606_02,2026-09-29 17:10:41,3394


In [ ]:
spark.stop()
# fresh(SANDBOX / ...)